In [1]:
!pip install -q PyPDF2 scikit-learn sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00


In [2]:
import os
import time
import json
from pathlib import Path

import PyPDF2
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:

from google.colab import drive
drive.mount('/content/drive')

DOCUMENTS_DIR = Path("/content/drive/MyDrive/rag/documents")  # adjust path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
print(list(DOCUMENTS_DIR.glob("*.pdf")))

[PosixPath('/content/drive/MyDrive/rag/documents/gestational_diabetes_diet.pdf'), PosixPath('/content/drive/MyDrive/rag/documents/iron_anemia_pregnancy.pdf'), PosixPath('/content/drive/MyDrive/rag/documents/food_safety_pregnancy.pdf'), PosixPath('/content/drive/MyDrive/rag/documents/general_pregnancy_nutrition.pdf'), PosixPath('/content/drive/MyDrive/rag/documents/calcium_vitamind_pregnancy.pdf')]


In [5]:
EVAL_SET = [
    {"query": "How many calories should I eat per trimester?", "expected_source": "general_pregnancy_nutrition.pdf"},
    {"query": "What foods provide balanced nutrition during pregnancy?", "expected_source": "general_pregnancy_nutrition.pdf"},
    {"query": "Recommended daily water intake while pregnant", "expected_source": "general_pregnancy_nutrition.pdf"},
    {"query": "How much iron do I need if I have anemia during pregnancy?", "expected_source": "iron_anemia_pregnancy.pdf"},
    {"query": "Best foods to treat iron deficiency in pregnancy", "expected_source": "iron_anemia_pregnancy.pdf"},
    {"query": "Symptoms of anemia during pregnancy", "expected_source": "iron_anemia_pregnancy.pdf"},
    {"query": "Is it safe to eat sushi while pregnant?", "expected_source": "food_safety_pregnancy.pdf"},
    {"query": "Which fish should I avoid during pregnancy?", "expected_source": "food_safety_pregnancy.pdf"},
    {"query": "Foods to avoid due to Listeria risk in pregnancy", "expected_source": "food_safety_pregnancy.pdf"},
    {"query": "How much calcium do I need daily during pregnancy?", "expected_source": "calcium_vitamind_pregnancy.pdf"},
    {"query": "Vitamin D sources for pregnant women", "expected_source": "calcium_vitamind_pregnancy.pdf"},
    {"query": "Recommended folate and calcium guidelines", "expected_source": "calcium_vitamind_pregnancy.pdf"},
    {"query": "What diet should I follow for gestational diabetes?", "expected_source": "gestational_diabetes_diet.pdf"},
    {"query": "Foods to avoid with gestational diabetes", "expected_source": "gestational_diabetes_diet.pdf"},
    {"query": "How to manage blood sugar levels during pregnancy", "expected_source": "gestational_diabetes_diet.pdf"},
]

TOP_K = 3

In [6]:
def load_and_chunk_documents():
    """Loads all PDFs in DOCUMENTS_DIR, returns list of (text_chunk, source_filename)."""
    chunks = []
    for pdf_path in sorted(DOCUMENTS_DIR.glob("*.pdf")):
        reader = PyPDF2.PdfReader(str(pdf_path))
        full_text = " ".join(page.extract_text() or "" for page in reader.pages)
        words = full_text.split()
        # ~180-word chunks with 30-word overlap
        chunk_size, overlap = 180, 30
        for i in range(0, len(words), chunk_size - overlap):
            chunk_words = words[i:i + chunk_size]
            if chunk_words:
                chunks.append((" ".join(chunk_words), pdf_path.name))
    return chunks

In [7]:
chunks = load_and_chunk_documents()
print(f"Loaded {len(chunks)} chunks from {len(list(DOCUMENTS_DIR.glob('*.pdf')))} documents.")

Loaded 19 chunks from 5 documents.


In [8]:
def _run_eval(embed_fn, chunk_vectors, sources, label):
    hits, reciprocal_ranks, latencies = [], [], []

    for item in EVAL_SET:
        start = time.perf_counter()
        query_vec = embed_fn(item["query"])
        sims = cosine_similarity(query_vec, chunk_vectors).flatten()
        elapsed_ms = (time.perf_counter() - start) * 1000

        top_indices = sims.argsort()[::-1][:TOP_K]
        retrieved_sources = [sources[i] for i in top_indices]

        hit = 1 if item["expected_source"] in retrieved_sources else 0
        if item["expected_source"] in retrieved_sources:
            rank = retrieved_sources.index(item["expected_source"]) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

        hits.append(hit)
        latencies.append(elapsed_ms)

    return {
        "label": label,
        "hit_rate_at_3": round(sum(hits) / len(hits), 3),
        "mrr": round(sum(reciprocal_ranks) / len(reciprocal_ranks), 3),
        "avg_latency_ms": round(sum(latencies) / len(latencies), 2),
        "n_queries": len(EVAL_SET),
    }

In [9]:
def evaluate_tfidf(chunks):
    """Baseline: TF-IDF retrieval (offline fallback method)."""
    texts = [c[0] for c in chunks]
    sources = [c[1] for c in chunks]

    vectorizer = TfidfVectorizer(stop_words="english")
    chunk_vectors = vectorizer.fit_transform(texts)

    return _run_eval(
        embed_fn=lambda q: vectorizer.transform([q]),
        chunk_vectors=chunk_vectors,
        sources=sources,
        label="TF-IDF (baseline / offline fallback)",
    )

tfidf_results = evaluate_tfidf(chunks)
tfidf_results

{'label': 'TF-IDF (baseline / offline fallback)',
 'hit_rate_at_3': 1.0,
 'mrr': 0.922,
 'avg_latency_ms': 1.71,
 'n_queries': 15}

In [10]:
from sentence_transformers import SentenceTransformer

def evaluate_minilm(chunks):
    """System: Sentence-Transformers MiniLM retrieval (primary method)."""
    texts = [c[0] for c in chunks]
    sources = [c[1] for c in chunks]

    model = SentenceTransformer("all-MiniLM-L6-v2")
    chunk_vectors = model.encode(texts)

    return _run_eval(
        embed_fn=lambda q: model.encode([q]),
        chunk_vectors=chunk_vectors,
        sources=sources,
        label="MiniLM (system / primary method)",
    )

try:
    minilm_results = evaluate_minilm(chunks)
except Exception as exc:
    print(f"MiniLM evaluation failed: {exc}")
    minilm_results = None

minilm_results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'label': 'MiniLM (system / primary method)',
 'hit_rate_at_3': 1.0,
 'mrr': 0.9,
 'avg_latency_ms': 42.01,
 'n_queries': 15}

In [11]:
print("=" * 60)
print("RESULTS")
print("=" * 60)
for result in [tfidf_results, minilm_results]:
    if result is None:
        continue
    print(f"\n{result['label']}")
    print(f"  Hit Rate@3:      {result['hit_rate_at_3']}")
    print(f"  MRR:             {result['mrr']}")
    print(f"  Avg Latency:     {result['avg_latency_ms']} ms")
    print(f"  Queries tested:  {result['n_queries']}")

output_path = Path("rag_evaluation_results.json")
with open(output_path, "w") as f:
    json.dump({"tfidf": tfidf_results, "minilm": minilm_results}, f, indent=2)
print(f"\nFull results saved to {output_path}")

RESULTS

TF-IDF (baseline / offline fallback)
  Hit Rate@3:      1.0
  MRR:             0.922
  Avg Latency:     1.71 ms
  Queries tested:  15

MiniLM (system / primary method)
  Hit Rate@3:      1.0
  MRR:             0.9
  Avg Latency:     42.01 ms
  Queries tested:  15

Full results saved to rag_evaluation_results.json


In [12]:
from google.colab import files
files.download("rag_evaluation_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>